In [7]:
!pip install pandas plotly kaleido
!pip install --upgrade kaleido
!pip install -U kaleido


In [3]:
import pandas as pd
import glob
import os
import plotly.graph_objects as go


In [2]:
ruta_archivos = 'results'


### Generation of plots for comparing learning curves

In [8]:

levels = ['simple', 'medium', 'complex']
algorithms = ['dqn', 'citsnn']
output_dir = 'figures_individuales'
os.makedirs(output_dir, exist_ok=True)

colors = {
    'DQN': 'blue',
    'CIT-SNN': 'green'
}

for level in levels:
    for i in range(1, 6):  # 5 experimentos
        fig = go.Figure()
        has_data = False

        for algo in algorithms:
            algo_label = 'DQN' if algo == 'dqn' else 'CIT-SNN'
            path = f"{ruta_archivos}/{algo}"
            file_path = os.path.join(path, f"{i}_learning_curves_{level}.csv")

            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                fig.add_trace(go.Scatter(
                    x=df['Episode'],
                    y=df['Reward'],
                    mode='lines',
                    name=algo_label,
                    line=dict(color=colors[algo_label])
                ))
                has_data = True
            else:
                print(f"⚠️ Archivo no encontrado: {file_path}")

        if has_data:
            fig.update_layout(
                title=None,
                xaxis_title='Episode',
                yaxis_title='Reward',
                font=dict(size=16),
                template='plotly_white',
                # legend=dict(x=0.05, y=0.95)
            )

            # Guardar como PDF (alta definición)
            output_file = f"{output_dir}/{i}_learning_curves_{level}.pdf"
            fig.write_image(output_file, width=900, height=600, scale=3)
            print(f"✅ Imagen guardada: {output_file}")


WARNING	Thread(Thread-15 (run)) choreographer.browser_async:browser_async.py:_close()- Resorting to unclean kill browser.


✅ Imagen guardada: figures_individuales/1_learning_curves_simple.pdf
✅ Imagen guardada: figures_individuales/2_learning_curves_simple.pdf
✅ Imagen guardada: figures_individuales/3_learning_curves_simple.pdf
✅ Imagen guardada: figures_individuales/4_learning_curves_simple.pdf
✅ Imagen guardada: figures_individuales/5_learning_curves_simple.pdf


WARNING	Thread(Thread-25 (run)) choreographer.browser_async:browser_async.py:_close()- Resorting to unclean kill browser.


✅ Imagen guardada: figures_individuales/1_learning_curves_medium.pdf
✅ Imagen guardada: figures_individuales/2_learning_curves_medium.pdf
✅ Imagen guardada: figures_individuales/3_learning_curves_medium.pdf


WARNING	Thread(Thread-31 (run)) choreographer.browser_async:browser_async.py:_close()- Resorting to unclean kill browser.


✅ Imagen guardada: figures_individuales/4_learning_curves_medium.pdf
✅ Imagen guardada: figures_individuales/5_learning_curves_medium.pdf
✅ Imagen guardada: figures_individuales/1_learning_curves_complex.pdf
✅ Imagen guardada: figures_individuales/2_learning_curves_complex.pdf
✅ Imagen guardada: figures_individuales/3_learning_curves_complex.pdf
✅ Imagen guardada: figures_individuales/4_learning_curves_complex.pdf
✅ Imagen guardada: figures_individuales/5_learning_curves_complex.pdf


## Resumen table for algoritmhs

In [ ]:
# Leer y concatenar todos los archivos, agregando una columna "Experimento"

# NOTE change final path for citsnn or dqn
ruta_archivos = 'results/dqn'

dataframes = []

# Leer una vez todos los archivos CSV que coincidan con el patrón
archivos = sorted(glob.glob(os.path.join(ruta_archivos, '*_learning_curves_*.csv')))

print(f"Archivos encontrados: {len(archivos)}")  # Debería imprimir 15

for archivo in archivos:
    df = pd.read_csv(archivo)
    df['Archivo'] = os.path.basename(archivo)
    dataframes.append(df)

# Concatenar todos
df_todos = pd.concat(dataframes, ignore_index=True)

# Verificar cantidad total de rewards
print("Total de rewards:", len(df_todos))  # ✅ Debe dar 15000

estadisticas_algoritmo = df_todos.groupby('Algorithm')['Reward'].describe()
estadisticas_algoritmo


Archivos encontrados: 15
Total de rewards: 13227


,count,mean,std,min,25%,50%,75%,max
Algorithm,,,,,,,,
DQN,13227.0,-4.260027,16.794222,-368.94,-11.32,2.9,8.22,9.46


## EVALUATION

In [ ]:

metricas = []
for algo in ['dqn', 'citsnn']:
    ruta_archivos = f"results/{algo}"

    archivos = sorted(glob.glob(os.path.join(ruta_archivos, '*_*_*_metrics_summary.csv')))
    for archivo in archivos:
        df = pd.read_csv(archivo)
        
        partes = os.path.basename(archivo).split('_')
        experimento = partes[0]
        algoritmo = partes[1]
        complejidad = partes[2]

        df['Experimento'] = experimento
        df['Algoritmo'] = algoritmo
        df['Complejidad'] = complejidad
        metricas.append(df)

# Unir todo
df_metricas = pd.concat(metricas, ignore_index=True)

# Verificar
df_metricas

,Metric,Value,Experimento,Algoritmo,Complejidad
0,Success Rate,0.00,1,DQN,complex
1,Average Reward,-27.72,1,DQN,complex
2,Average Steps per Episode,462.00,1,DQN,complex
3,Convergence Speed (Episode),0.00,1,DQN,complex
4,Experiment,1.00,1,DQN,complex
...,...,...,...,...,...
145,Success Rate,1.00,5,CIT-SSN,simple
146,Average Reward,9.46,5,CIT-SSN,simple
147,Average Steps per Episode,10.00,5,CIT-SSN,simple
148,Convergence Speed (Episode),1.00,5,CIT-SSN,simple


In [33]:
tabla_resumen = df_metricas.pivot_table(index=['Complejidad', 'Algoritmo', 'Experimento'],
                                        columns='Metric',
                                        values='Value').reset_index()

tabla_resumen = tabla_resumen.round(2)

tabla_resumen[['Complejidad', 'Experimento', 'Average Reward', 'Average Steps per Episode', 'Convergence Speed (Episode)', 'Success Rate']]

Metric,Complejidad,Experimento,Average Reward,Average Steps per Episode,Convergence Speed (Episode),Success Rate
0,complex,1,8.74,22.0,1.0,1.0
1,complex,2,8.74,22.0,1.0,1.0
2,complex,3,8.74,22.0,1.0,1.0
3,complex,4,8.74,22.0,1.0,1.0
4,complex,5,8.74,22.0,1.0,1.0
5,complex,1,-27.72,462.0,0.0,0.0
6,complex,2,8.74,22.0,1.0,1.0
7,complex,3,8.74,22.0,1.0,1.0
8,complex,4,-27.72,462.0,0.0,0.0
9,complex,5,-27.72,462.0,0.0,0.0


In [34]:
print(tabla_resumen['Average Reward'].mean())
print(tabla_resumen['Average Steps per Episode'].mean())
print(tabla_resumen['Convergence Speed (Episode)'].mean())

-0.29999999999999954
105.6
0.6


In [37]:
tabla_promedios = df_metricas.pivot_table(
    index=['Complejidad', 'Algoritmo'],
    columns='Metric',
    values='Value',
    aggfunc='mean'
).reset_index()

# Redondear los valores
tabla_promedios = tabla_promedios.round(2)

# Mostrar la tabla ordenada
tabla_promedios = tabla_promedios[['Complejidad', 'Algoritmo', 
                                   'Average Reward', 
                                   'Average Steps per Episode',
                                   'Convergence Speed (Episode)', 
                                   'Success Rate']]

tabla_promedios

Metric,Complejidad,Algoritmo,Average Reward,Average Steps per Episode,Convergence Speed (Episode),Success Rate
0,complex,CIT-SSN,8.74,22.0,1.0,1.0
1,complex,DQN,-13.14,286.0,0.4,0.4
2,medium,CIT-SSN,9.22,14.0,1.0,1.0
3,medium,DQN,-8.52,175.6,0.2,0.2
4,simple,CIT-SSN,9.46,10.0,1.0,1.0
5,simple,DQN,-7.56,126.0,0.0,0.0


### ANOVA

In [4]:
from scipy.stats import f_oneway

for algo in ['dqn', 'citsnn']:
    ruta_archivos = f"results/{algo}"
    archivos = sorted(glob.glob(os.path.join(ruta_archivos, '*_*_*_metrics.csv')))

    datos = []

    # Leer y procesar cada archivo
    for archivo in archivos:
        df = pd.read_csv(archivo)

        # Extraer metadatos del nombre del archivo
        nombre = os.path.basename(archivo)
        partes = nombre.split('_')
        experimento = int(partes[0])
        algoritmo = partes[1]
        complejidad = partes[2]  # Ej: simple, medium, complex

        # Agregar columnas de contexto
        df['Experimento'] = experimento
        df['Algoritmo'] = algoritmo
        df['Complejidad'] = complejidad

        datos.append(df)

    # 👉 Aquí unimos todo fuera del loop
    df_todos = pd.concat(datos, ignore_index=True)
    df_todos['Reward'] = pd.to_numeric(df_todos['Reward'], errors='coerce')

    # Agrupar por nivel de complejidad
    grupos = df_todos.groupby('Complejidad')['Reward']

    # Convertir a listas por grupo (solo si hay al menos 2 grupos)
    valores_por_grupo = [grupo.dropna().tolist() for _, grupo in grupos if len(grupo.dropna()) > 0]

    if len(valores_por_grupo) >= 2:
        # ANOVA
        f_stat, p_value = f_oneway(*valores_por_grupo)

        print(f"\nAlgoritmo: {algo}")
        print(f"F-statistic: {f_stat:.4f}")
        print(f"p-value: {p_value:.4e}")

        if p_value < 0.05:
            print("✅ Hay diferencias estadísticamente significativas entre los grupos.")
        else:
            print("❌ No hay diferencias significativas entre los grupos.")
    else:
        print(f"\nAlgoritmo: {algo} - ❌ No hay suficientes grupos para aplicar ANOVA.")


Algoritmo: dqn
F-statistic: 3.3706
p-value: 3.4648e-02
✅ Hay diferencias estadísticamente significativas entre los grupos.

Algoritmo: citsnn
F-statistic: inf
p-value: 0.0000e+00
✅ Hay diferencias estadísticamente significativas entre los grupos.


c:\Users\LENOVO\Desktop\Documents\Dropbox\UD\PHD\Project\comparative_transfer_study\venv\lib\site-packages\scipy\stats\_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


In [5]:
for algo in ['dqn', 'citsnn']:
    df_algo = df_todos[df_todos['Algoritmo'].str.lower() == algo.lower()]
    df_algo = df_algo.dropna(subset=['Reward'])  # eliminar NaNs si hay

    # Total de observaciones (N) y número de grupos (k)
    N = len(df_algo)
    k = df_algo['Complejidad'].nunique()

    # Grados de libertad
    gl_entre = k - 1
    gl_dentro = N - k
    gl_total = N - 1

    print(f"\n📊 Algoritmo: {algo.upper()}")
    print(f"Grupos de complejidad: {k}")
    print(f"Total de observaciones (episodios): {N}")
    print(f"Grados de libertad entre grupos (GL entre): {gl_entre}")
    print(f"Grados de libertad dentro de grupos (GL dentro): {gl_dentro}")
    print(f"Grados de libertad total: {gl_total}")


📊 Algoritmo: DQN
Grupos de complejidad: 0
Total de observaciones (episodios): 0
Grados de libertad entre grupos (GL entre): -1
Grados de libertad dentro de grupos (GL dentro): 0
Grados de libertad total: -1

📊 Algoritmo: CITSNN
Grupos de complejidad: 0
Total de observaciones (episodios): 0
Grados de libertad entre grupos (GL entre): -1
Grados de libertad dentro de grupos (GL dentro): 0
Grados de libertad total: -1
